In [21]:
import psycopg2
from psycopg2 import Error

def create_db_connection(host_name, port_name, user_name, user_password, dbname):
    connection = None
    try:
        connection = psycopg2.connect(dbname=dbname, user=user_name, password=user_password, host=host_name, port=port_name)
        print("openGauss Database connection successful")
    except Error as err:
        print(f"Error: {err}")
    return connection

In [22]:
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

openGauss Database connection successful


In [23]:
def read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as err:
        print(f"Error: '{err}'")

In [24]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as err:
        print(f"Error: {err}")

In [25]:
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")
connection.set_client_encoding('UTF8')

cursor = connection.cursor()
try:
    with open('school.sql', 'r', encoding='utf-8') as file:
        sql_script = file.read()
except UnicodeDecodeError:
    with open('school.sql', 'r', encoding='gb18030') as file:
        sql_script = file.read()

sql_commands = sql_script.split(';')
for command in sql_commands:
    cmd = command.strip()
    if cmd:
        try:
            cursor.execute(cmd)
        except Exception as e:
            print(f"执行错误: {e}")
            print(f"问题SQL: {cmd}")

print("school.sql 执行成功！")
connection.commit()
connection.close()

openGauss Database connection successful
school.sql 执行成功！


In [32]:
#(1) 创建模式 sche1，然后创建表空间 example1，创建分区表 sche1.students(包括学号 id，姓名，年龄)，并基于年龄（<18、18~20、20~25、25~40），将分区表划分 4 个分区 P1、P2、P3、P4。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

connection.autocommit = True

create_schema_query = "CREATE SCHEMA IF NOT EXISTS sche1;"
execute_query(connection, create_schema_query)

create_tablespace_query = "CREATE TABLESPACE example1 RELATIVE LOCATION 'tablespace/example1';"
execute_query(connection, create_tablespace_query)

connection.autocommit = False

create_table_query = """
CREATE TABLE sche1.students (
    id VARCHAR(20) NOT NULL,
    name VARCHAR(50), 
    age INT 
) 
PARTITION BY RANGE (age) 
(
    PARTITION P1 VALUES LESS THAN (18) TABLESPACE example1, 
    PARTITION P2 VALUES LESS THAN (20) TABLESPACE example1, 
    PARTITION P3 VALUES LESS THAN (25) TABLESPACE example1, 
    PARTITION P4 VALUES LESS THAN (40) TABLESPACE example1
);
"""
execute_query(connection, create_table_query)

if connection:
    connection.close()
    print("PostgreSQL connection is closed")

openGauss Database connection successful
Query executed successfully
Query executed successfully
Query executed successfully
PostgreSQL connection is closed


In [43]:
#(1) 向分区表 sche1.students 中加入增加一些记录(“1001”“, aerf”,10)、(“1021”“, beu”,19)、(“1031”,“cekf”,11)；
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

insert_query = """
INSERT INTO sche1.students (id, name, age) VALUES
('1001', 'aerf', 10), 
('1021', 'beu', 19), 
('1031', 'cekf', 11);
"""
execute_query(connection, insert_query)
connection.commit() 

if connection:
    connection.close()
    print("PostgreSQL connection is closed")

openGauss Database connection successful
Query executed successfully
PostgreSQL connection is closed


In [46]:
#(2) 查询分区 P1 的所有信息【比较使用分区表和不使用分区表（<18）的查询时间比较】；
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# 查询分区P1的所有记录（使用分区表）
print("查询分区P1的所有记录（使用分区表）：")
partition_query = """SELECT * FROM sche1.students PARTITION(P1);"""
partition_result = read_query(connection, partition_query)
for row in partition_result:
    print(row)

# 创建相同的非分区表用于对比
create_non_partitioned_table_query = """
CREATE TABLE sche1.students_non_partitioned (
    id VARCHAR(20),
    name VARCHAR(50),
    age INT
) TABLESPACE example1;
"""
execute_query(connection, create_non_partitioned_table_query)

# 将分区表的数据插入到非分区表中
insert_non_partitioned_data_query = "INSERT INTO sche1.students_non_partitioned SELECT * FROM sche1.students;"
execute_query(connection, insert_non_partitioned_data_query)

# 查询非分区表中年龄小于18的记录
print("\n查询非分区表中年龄小于18的记录：")
non_partition_query = "SELECT * FROM sche1.students_non_partitioned WHERE age < 18;"
non_partition_result = read_query(connection, non_partition_query)
for row in non_partition_result:
    print(row)

# 分析分区表查询（P1分区）
print("\n分区表查询（P1分区）的分析结果：")
partition_analyze_query = "EXPLAIN ANALYZE SELECT * FROM sche1.students PARTITION(P1);"
partition_analyze_result = read_query(connection, partition_analyze_query)
for row in partition_analyze_result:
    print(row[0])

# 分析非分区表等价查询（age < 18）
print("\n非分区表等价查询（age < 18）的分析结果：")
non_partition_analyze_query = "EXPLAIN ANALYZE SELECT * FROM sche1.students_non_partitioned WHERE age < 18;"
non_partition_analyze_result = read_query(connection, non_partition_analyze_query)
for row in non_partition_analyze_result:
    print(row[0])

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
查询分区P1的所有记录（使用分区表）：
('1001', 'aerf', 10)
('1031', 'cekf', 11)
Query executed successfully
Query executed successfully

查询非分区表中年龄小于18的记录：
('1001', 'aerf', 10)
('1031', 'cekf', 11)

分区表查询（P1分区）的分析结果：
Partition Iterator  (cost=0.00..10.09 rows=809 width=71) (actual time=0.089..0.091 rows=2 loops=1)
  Iterations: 1
  ->  Partitioned Seq Scan on students  (cost=0.00..10.09 rows=809 width=71) (actual time=0.022..0.024 rows=2 loops=1)
        Selected Partitions:  1
Total runtime: 1.485 ms

非分区表等价查询（age < 18）的分析结果：
Seq Scan on students_non_partitioned  (cost=0.00..20.11 rows=270 width=71) (actual time=0.050..0.055 rows=2 loops=1)
  Filter: (age < 18)
  Rows Removed by Filter: 1
Total runtime: 0.760 ms
Database connection closed


In [48]:
#(3) 删除分区表 sche1.students 和表空间 example1，并删除模式 sche1。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# 删除分区表
drop_table_query = "DROP TABLE IF EXISTS sche1.students CASCADE;"
execute_query(connection, drop_table_query)

# 查询使用指定表空间的对象
check_tablespace_query = """
SELECT n.nspname, c.relname, 
       CASE c.relkind 
         WHEN 'r' THEN 'table' 
         WHEN 'i' THEN 'index'
         ELSE 'other'
       END as type
FROM pg_class c
JOIN pg_namespace n ON c.relnamespace = n.oid
WHERE c.reltablespace = (SELECT oid FROM pg_tablespace WHERE spcname = 'example1');
"""
result = execute_query(connection, check_tablespace_query)
if result:
    print("使用指定表空间的对象:")
    for row in result:
        print(row)
else:
    print("未找到使用指定表空间的对象。")

# 删除非分区表
drop_non_partitioned_table_query = "DROP TABLE IF EXISTS sche1.students_non_partitioned CASCADE;"
execute_query(connection, drop_non_partitioned_table_query)

connection.autocommit = True
# 删除表空间
drop_tablespace_query = "DROP TABLESPACE IF EXISTS example1;"
execute_query(connection, drop_tablespace_query)

# 删除模式
drop_schema_query = "DROP SCHEMA IF EXISTS sche1 CASCADE;"
execute_query(connection, drop_schema_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Query executed successfully
未找到使用指定表空间的对象。
Query executed successfully
Query executed successfully
Query executed successfully
Database connection closed


In [49]:
#(1) 找出学号为“1437120165”的同学，将她的学号更新为“1001”，并更新和参照外键数据；
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
BEGIN; 
INSERT INTO xs(xh, xm, ydh, bj, chrq, xb)
SELECT '1001', xm, ydh, bj, chrq, xb FROM xs WHERE xh = '1437120165';
UPDATE xk SET xh = '1001' WHERE xh = '1437120165';
DELETE FROM xs WHERE xh = '1437120165';
COMMIT;
"""
execute_query(connection, update_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [50]:
#(2) 学号更新完之后，删除学号为“1001”同学的相关信息。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
BEGIN; 
DELETE FROM xk WHERE xh = '1001';
DELETE FROM xs WHERE xh = '1001';
COMMIT; 
"""
execute_query(connection, update_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [52]:
#(1) 寻找性别为“NULL”的数据，并将其赋值为“男”；
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
UPDATE xs 
SET xb = '男' 
WHERE xb IS NULL;
"""
execute_query(connection, update_query.encode(encoding='utf-8'))

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [54]:
#(2) 寻找出生日期为“NULL”的数据，删除这些同学的信息，以及他们的选课信息。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
BEGIN; 
DELETE FROM xk 
WHERE xh IN (SELECT xh FROM xs WHERE chrq IS NULL);
DELETE FROM xs 
WHERE chrq IS NULL;
COMMIT; 
"""
execute_query(connection, update_query.encode(encoding='utf-8'))

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [55]:
#(1) 删除设计与艺术学院（zy）的相关信息，并把属于设计与艺术学院的同学，老师，授课信息以及同学的选课信息删除。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
BEGIN; 
DELETE FROM xk
WHERE xh IN (SELECT xh FROM xs WHERE ydh = 'zy') 
   OR (kcbh, jsbh) IN (SELECT kcbh, bh FROM sk WHERE bh IN (SELECT jsbh FROM js WHERE ydh = 'zy')); 
DELETE FROM sk
WHERE bh IN (SELECT jsbh FROM js WHERE ydh = 'zy');
DELETE FROM xs
WHERE ydh = 'zy';
DELETE FROM js
WHERE ydh = 'zy';
DELETE FROM xyb
WHERE ydh = 'zy';
COMMIT; 
"""
execute_query(connection, update_query.encode(encoding='utf-8'))

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [56]:
#(1) 找出挂过科的同学（至少有一名课程成绩在小于 60），将他们的班级信息更新为“08012048”；
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
UPDATE xs
SET bj = '08012048'
WHERE xh IN (
    SELECT DISTINCT xh 
    FROM xk 
    WHERE cj < 60
);
"""
execute_query(connection, update_query.encode(encoding='utf-8'))

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [57]:
#(2) 找出挂过科的同学，并删除他们的对应数据；与此同时，对应的选课信息也被删除。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

update_query = """
BEGIN; 
DELETE FROM xk
WHERE xh IN (
    SELECT DISTINCT xh 
    FROM xk 
    WHERE cj < 60
);
DELETE FROM xs
WHERE xh IN (
    SELECT DISTINCT xh 
    FROM xk 
    WHERE cj < 60
);
COMMIT; 
"""
execute_query(connection, update_query.encode(encoding='utf-8'))

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Database connection closed


In [61]:
#1、使用 create index 创建索引
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")
def query_index_info(connection, table_name):
    cursor = connection.cursor()
    try:
        query = f"""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = '{table_name}';
        """
        cursor.execute(query)
        result = cursor.fetchall()
        print(f"索引信息（表: {table_name}）:")
        for row in result:
            print(f"索引名: {row[0]}, 定义: {row[1]}")
    except Error as err:
        print(f"Error querying index information: {err}")
    finally:
        cursor.close()
        
# (1) 对学生表(xs)中的学号(xh)列创建单列索引 stu_index
create_stu_index_query = "CREATE INDEX stu_index ON xs(xh);"
execute_query(connection, create_stu_index_query)

# (2) 对学生表(xs)中的姓名(xm)和班级(bj)列创建复合索引 ad_index
create_ad_index_query = "CREATE INDEX ad_index ON xs(xm, bj);"
execute_query(connection, create_ad_index_query)

# (3) 对学生表(xs)中的出生日期(chrq)（00/00/0000）的年份创建表达式索引
create_chrq_year_index_query = "CREATE INDEX idx_chrq_year ON xs(EXTRACT(YEAR FROM chrq));"
execute_query(connection, create_chrq_year_index_query)

# (4) 对学院表(xyb)中的学院编号(ydh)列创建唯一索引
create_xyb_ydh_unique_index_query = "CREATE UNIQUE INDEX idx_xyb_ydh_unique ON xyb(ydh);"
execute_query(connection, create_xyb_ydh_unique_index_query)

# (5) 对学院表(xyb)中的学院编号(ydh)列创建部分索引（只索引单号的部门 ID）
create_xyb_odd_ydh_index_query = """
CREATE INDEX idx_xyb_odd_ydh ON xyb(ydh)
WHERE (CAST(SUBSTRING(ydh FROM '[0-9]+$') AS INTEGER) % 2) = 1;
"""
execute_query(connection, create_xyb_odd_ydh_index_query)

# 查询学生表(xs)的索引信息
query_index_info(connection, "xs")

# 查询学院表(xyb)的索引信息
query_index_info(connection, "xyb")

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
索引信息（表: xs）:
索引名: idx_chrq_year, 定义: CREATE INDEX idx_chrq_year ON xs USING btree (date_part('year'::text, chrq)) TABLESPACE pg_default
索引名: ad_index, 定义: CREATE INDEX ad_index ON xs USING btree (xm, bj) TABLESPACE pg_default
索引名: stu_index, 定义: CREATE INDEX stu_index ON xs USING btree (xh) TABLESPACE pg_default
索引名: xs_pkey, 定义: CREATE UNIQUE INDEX xs_pkey ON xs USING btree (xh) TABLESPACE pg_default
索引信息（表: xyb）:
索引名: idx_xyb_odd_ydh, 定义: CREATE INDEX idx_xyb_odd_ydh ON xyb USING btree (ydh) TABLESPACE pg_default WHERE ((("substring"((ydh)::text, '[0-9]+$'::text))::integer % 2) = 1)
索引名: idx_xyb_ydh_unique, 定义: CREATE UNIQUE INDEX idx_xyb_ydh_unique ON xyb USING btree (ydh) TABLESPACE pg_default
索引名: xyb_pkey, 定义: CREATE UNIQUE INDEX xyb_pkey ON xyb USING btree (ydh) TABLESPACE pg_default
Database connecti

In [62]:
#2、使用 alter table 添加索引
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# (1) 对学生表(xs)中的出生日期列添加一个唯一索引 date_index，姓名(xm)和性别(xb)列添加一个复合索引 name_sex_index；
add_date_index_query = "ALTER TABLE xs ADD CONSTRAINT date_index UNIQUE (chrq);"
execute_query(connection, add_date_index_query)

create_name_sex_index_query = "CREATE INDEX name_sex_index ON xs(xm, xb);"
execute_query(connection, create_name_sex_index_query)

# (2) 对学生选课表(xk)中的教师编号(jsbh)列创建外键索引
create_fk_jsbh_idx_query = "CREATE INDEX fk_jsbh_idx ON xk(jsbh);"
execute_query(connection, create_fk_jsbh_idx_query)

# 查询学生表(xs)的索引信息
query_index_info(connection, "xs")

# 查询学生选课表(xk)的索引信息
query_index_info(connection, "xk")

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Query executed successfully
Query executed successfully
索引信息（表: xs）:
索引名: name_sex_index, 定义: CREATE INDEX name_sex_index ON xs USING btree (xm, xb) TABLESPACE pg_default
索引名: date_index, 定义: CREATE UNIQUE INDEX date_index ON xs USING btree (chrq) TABLESPACE pg_default
索引名: idx_chrq_year, 定义: CREATE INDEX idx_chrq_year ON xs USING btree (date_part('year'::text, chrq)) TABLESPACE pg_default
索引名: ad_index, 定义: CREATE INDEX ad_index ON xs USING btree (xm, bj) TABLESPACE pg_default
索引名: stu_index, 定义: CREATE INDEX stu_index ON xs USING btree (xh) TABLESPACE pg_default
索引名: xs_pkey, 定义: CREATE UNIQUE INDEX xs_pkey ON xs USING btree (xh) TABLESPACE pg_default
索引信息（表: xk）:
索引名: fk_jsbh_idx, 定义: CREATE INDEX fk_jsbh_idx ON xk USING btree (jsbh) TABLESPACE pg_default
索引名: xk_pkey, 定义: CREATE UNIQUE INDEX xk_pkey ON xk USING btree (xh, kcbh, jsbh) TABLESPACE pg_default
Database connection closed


In [63]:
#3、在创建表的同时创建索引
#(1) 创建 game 表（比赛编号，比赛名称、比赛时间、学分）（每列的数据类型及长度等信息自定），并对比赛编号列创建主键索引game_pkey，在学分列创建唯一索引game_cre_index。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

create_game_table_query = """
CREATE TABLE game (
    game_id VARCHAR(10) NOT NULL, 
    game_name VARCHAR(50) NOT NULL, 
    game_time TIMESTAMP, 
    credit DECIMAL(3,1), 
    CONSTRAINT game_pkey PRIMARY KEY (game_id),
    CONSTRAINT game_cre_index UNIQUE (credit)
);
"""
execute_query(connection, create_game_table_query)
# 查询 game 表的索引信息
query_index_info(connection, "game")

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
索引信息（表: game）:
索引名: game_cre_index, 定义: CREATE UNIQUE INDEX game_cre_index ON game USING btree (credit) TABLESPACE pg_default
索引名: game_pkey, 定义: CREATE UNIQUE INDEX game_pkey ON game USING btree (game_id) TABLESPACE pg_default
Database connection closed


In [64]:
#4、 查询计划
#(1) 通过查询计划（EXPLAIN ANALYZE）查询学生表中出生年份在 1998 的学生信息。【比较使用索引和不使用索引的查询时间比较】
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

create_index_query = "CREATE INDEX IF NOT EXISTS idx_xs_birth_year ON xs(EXTRACT(YEAR FROM chrq));"
execute_query(connection, create_index_query)

# 测试1: 不使用索引
print("不使用索引的查询计划：")
disable_indexes_query = """
SET enable_indexscan = off;
SET enable_bitmapscan = off;
"""
execute_query(connection, disable_indexes_query)
explain_without_index_query = "EXPLAIN ANALYZE SELECT xh, xm, chrq FROM xs WHERE EXTRACT(YEAR FROM chrq) = 1998;"
non_partition_analyze_result = read_query(connection, explain_without_index_query)
if non_partition_analyze_result:
    for row in non_partition_analyze_result:
        print(row[0])

# 测试2: 使用索引
print("\n使用索引的查询计划：")
enable_indexes_query = """
SET enable_indexscan = on;
SET enable_bitmapscan = on;
"""
execute_query(connection, enable_indexes_query)
explain_with_index_query = "EXPLAIN ANALYZE SELECT xh, xm, chrq FROM xs WHERE EXTRACT(YEAR FROM chrq) = 1998;"
partition_analyze_result = read_query(connection, explain_with_index_query)
if partition_analyze_result:
    for row in partition_analyze_result:
        print(row[0])

# 重置设置
reset_settings_query = """
RESET enable_indexscan;
RESET enable_bitmapscan;
"""
execute_query(connection, reset_settings_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
不使用索引的查询计划：
Query executed successfully
Seq Scan on xs  (cost=0.00..1.60 rows=1 width=26) (actual time=0.044..0.058 rows=3 loops=1)
  Filter: (date_part('year'::text, chrq) = 1998::double precision)
  Rows Removed by Filter: 37
Total runtime: 0.289 ms

使用索引的查询计划：
Query executed successfully
Seq Scan on xs  (cost=0.00..1.60 rows=1 width=26) (actual time=0.023..0.036 rows=3 loops=1)
  Filter: (date_part('year'::text, chrq) = 1998::double precision)
  Rows Removed by Filter: 37
Total runtime: 0.148 ms
Query executed successfully
Database connection closed


In [65]:
#5、 删除索引
#(1) 使用 drop index 删除索引 stu_index、ad_index。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

drop_stu_index_query = "DROP INDEX IF EXISTS stu_index;"
execute_query(connection, drop_stu_index_query)

drop_ad_index_query = "DROP INDEX IF EXISTS ad_index;"
execute_query(connection, drop_ad_index_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
Query executed successfully
Database connection closed


In [66]:
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")
connection.set_client_encoding('UTF8')

cursor = connection.cursor()
try:
    with open('employ.sql', 'r', encoding='utf-8') as file:
        sql_script = file.read()
except UnicodeDecodeError:
    with open('employ.sql', 'r', encoding='gb18030') as file:
        sql_script = file.read()

sql_commands = sql_script.split(';')
for command in sql_commands:
    cmd = command.strip()
    if cmd:
        try:
            cursor.execute(cmd)
        except Exception as e:
            print(f"执行错误: {e}")
            print(f"问题SQL: {cmd}")

print("employ.sql 执行成功！")
connection.commit()
connection.close()

openGauss Database connection successful
employ.sql 执行成功！


In [74]:
#1、基于分区表查询【体现“未分区”和“分区后”】 
#(1) 查询 salaries 表中 2001 年所有员工的薪资总和。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# 查询 salaries 表中 2001 年所有员工的薪资总和
query_1 = """
SELECT SUM(salary) AS total_salary_2001
FROM kk.salaries
WHERE from_date >= '2001-01-01' AND to_date <= '2001-12-31';
"""
result_1 = read_query(connection, query_1)
print("salaries 表中 2001 年所有员工的薪资总和:")
for row in result_1:
    print(row[0])

# 分析查询 salaries 表中 2001 年所有员工的薪资总和的查询计划
query_2 = """
EXPLAIN ANALYZE
SELECT SUM(salary) AS total_salary_2001
FROM kk.salaries
WHERE from_date >= '2001-01-01' AND to_date <= '2001-12-31';
"""
result_2 = read_query(connection, query_2)
print("\nsalaries 表中 2001 年薪资总和查询的查询计划:")
for row in result_2:
    print(row[0])

# 创建分区表
query_3 = """
CREATE TABLE kk.salaries_partitioned (
    emp_no      INT             NOT NULL,
    salary      INT             NOT NULL,
    from_date   DATE            NOT NULL,
    to_date     DATE            NOT NULL,
    FOREIGN KEY (emp_no) REFERENCES kk.employees (emp_no) ON DELETE CASCADE,
    PRIMARY KEY (emp_no, from_date)
)
PARTITION BY RANGE (from_date) (
    PARTITION salaries_2000 VALUES LESS THAN ('2001-01-01'),
    PARTITION salaries_2001 VALUES LESS THAN ('2002-01-01'),
    PARTITION salaries_2002 VALUES LESS THAN ('2003-01-01'),
    PARTITION salaries_future VALUES LESS THAN (MAXVALUE)
);
"""
execute_query(connection, query_3)

# 将数据插入分区表
query_4 = "INSERT INTO kk.salaries_partitioned SELECT * FROM kk.salaries;"
execute_query(connection, query_4)

# 查询分区表中 2001 年所有员工的薪资总和
query_5 = """
SELECT SUM(salary) AS total_salary_2001
FROM kk.salaries_partitioned PARTITION(salaries_2001)
WHERE from_date >= '2001-01-01' AND to_date <= '2001-12-31';
"""
result_5 = read_query(connection, query_5)
print("\n分区表中 2001 年所有员工的薪资总和:")
for row in result_5:
    print(row[0])

# 分析查询分区表中 2001 年所有员工的薪资总和的查询计划
query_6 = """
EXPLAIN ANALYZE
SELECT SUM(salary) FROM kk.salaries_partitioned
WHERE from_date >= '2001-01-01' AND from_date < '2002-01-01';
"""
result_6 = read_query(connection, query_6)
print("\n分区表中 2001 年薪资总和查询的查询计划:")
for row in result_6:
    print(row[0])
    
if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
salaries 表中 2001 年所有员工的薪资总和:
232389564

salaries 表中 2001 年薪资总和查询的查询计划:
Aggregate  (cost=61746.11..61746.12 rows=1 width=12) (actual time=461.199..461.199 rows=1 loops=1)
  ->  Seq Scan on salaries  (cost=0.00..60892.71 rows=341363 width=4) (actual time=0.209..460.377 rows=3517 loops=1)
        Filter: ((from_date >= '2001-01-01 00:00:00'::timestamp without time zone) AND (to_date <= '2001-12-31 00:00:00'::timestamp without time zone))
        Rows Removed by Filter: 2840530
Total runtime: 461.420 ms
Query executed successfully
Query executed successfully

分区表中 2001 年所有员工的薪资总和:
232389564

分区表中 2001 年薪资总和查询的查询计划:
Aggregate  (cost=43338.12..43338.13 rows=1 width=12) (actual time=300.742..300.743 rows=1 loops=1)
  ->  Partition Iterator  (cost=38538.80..43303.65 rows=13790 width=4) (actual time=156.068..248.778 rows=247652 loops=1)
        Iterations: 1
        ->  Partitioned Bitmap Heap Scan on salaries_partitioned  (cost=38538.80..43303.65 rows=1

In [78]:
#2、基于索引查询【体现“无索引”和“有索引”】 
#(1) 在 dept_emp 表中，查询在编号为'd001'的部门的员工任职记录。【单列索引】 
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

 # 禁用可能的缓存以确保公平比较
disable_cache_query = """
SET enable_seqscan = on;
SET enable_indexscan = off;
"""
execute_query(connection, disable_cache_query)

# 查询部门 'd001' 的所有员工任职记录（无索引）
no_index_query = """
EXPLAIN ANALYZE
SELECT * FROM kk.dept_emp 
WHERE dept_no = 'd001';
"""
no_index_result = read_query(connection, no_index_query)
print("无索引时查询部门 'd001' 的员工任职记录的查询计划:")
for row in no_index_result:
    print(row[0])

count_no_index_query = "SELECT count(*) FROM kk.dept_emp WHERE dept_no = 'd001';"
count_no_index_result = read_query(connection, count_no_index_query)
print("无索引时部门 'd001' 的员工任职记录数量:", count_no_index_result[0][0])

# 创建索引
create_index_query = "CREATE INDEX idx_dept_emp_dept_no ON kk.dept_emp(dept_no);"
execute_query(connection, create_index_query)

# 结束当前事务
connection.commit()

# 开启自动提交模式
connection.autocommit = True

# 分析表
analyze_query = "ANALYZE kk.dept_emp;"
execute_query(connection, analyze_query)

# 关闭自动提交模式
connection.autocommit = False

# 启用索引扫描
enable_index_query = """
SET enable_seqscan = off;
SET enable_indexscan = on;
"""
execute_query(connection, enable_index_query)

# 查询部门 'd001' 的所有员工任职记录（有索引）
with_index_query = """
EXPLAIN ANALYZE
SELECT * FROM kk.dept_emp
WHERE dept_no = 'd001';
"""
with_index_result = read_query(connection, with_index_query)
print("\n有索引时查询部门 'd001' 的员工任职记录的查询计划:")
for row in with_index_result:
    print(row[0])

count_with_index_query = "SELECT count(*) FROM kk.dept_emp WHERE dept_no = 'd001';"
count_with_index_result = read_query(connection, count_with_index_query)
print("有索引时部门 'd001' 的员工任职记录数量:", count_with_index_result[0][0])

# 删除索引
drop_index_query = "DROP INDEX IF EXISTS kk.idx_dept_emp_dept_no;"
execute_query(connection, drop_index_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
无索引时查询部门 'd001' 的员工任职记录的查询计划:
Seq Scan on dept_emp  (cost=0.00..6602.04 rows=20117 width=25) (actual time=0.063..60.956 rows=20211 loops=1)
  Filter: (dept_no = 'd001'::bpchar)
  Rows Removed by Filter: 311392
Total runtime: 63.128 ms
无索引时部门 'd001' 的员工任职记录数量: 20211
Query executed successfully
Query executed successfully
Query executed successfully

有索引时查询部门 'd001' 的员工任职记录的查询计划:
Bitmap Heap Scan on dept_emp  (cost=387.24..3100.68 rows=20515 width=25) (actual time=2.696..9.340 rows=20211 loops=1)
  Recheck Cond: (dept_no = 'd001'::bpchar)
  Heap Blocks: exact=2457
  ->  Bitmap Index Scan on idx_dept_emp_dept_no  (cost=0.00..382.11 rows=20515 width=0) (actual time=2.348..2.348 rows=20211 loops=1)
        Index Cond: (dept_no = 'd001'::bpchar)
Total runtime: 10.738 ms
有索引时部门 'd001' 的员工任职记录数量: 20211
Query executed successfully
Database connection closed


In [80]:
#(2) 在 salaries 表中，查询薪资高于 100000 的记录。【部分索引】 
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# 禁用索引扫描，强制全表扫描
disable_index_scan_query = """
SET enable_indexscan = off;
SET enable_bitmapscan = off;
"""
execute_query(connection, disable_index_scan_query)

# 查询薪资高于 100000 的记录数量
count_high_salary_query = "SELECT count(*) AS high_salary_count FROM kk.salaries WHERE salary > 100000;"
count_high_salary_result = read_query(connection, count_high_salary_query)
print("禁用索引扫描时，薪资高于 100000 的记录数量:", count_high_salary_result[0][0])

# 分析全表扫描的查询计划
explain_full_scan_query = "EXPLAIN ANALYZE SELECT * FROM kk.salaries WHERE salary > 100000;"
explain_full_scan_result = read_query(connection, explain_full_scan_query)
print("\n禁用索引扫描时，查询薪资高于 100000 的记录的查询计划:")
for row in explain_full_scan_result:
    print(row[0])

# 创建部分索引
create_partial_index_query = "CREATE INDEX idx_salaries_high_salary ON kk.salaries(salary) WHERE salary > 100000;"
execute_query(connection, create_partial_index_query)

# 结束当前事务
connection.commit()

# 开启自动提交模式
connection.autocommit = True
# 分析表
analyze_table_query = "ANALYZE kk.salaries;"
execute_query(connection, analyze_table_query)
# 关闭自动提交模式
connection.autocommit = False
# 启用索引扫描
enable_index_scan_query = """
SET enable_indexscan = on;
SET enable_bitmapscan = on;
"""
execute_query(connection, enable_index_scan_query)

# 再次查询薪资高于 100000 的记录数量
count_high_salary_query_after_index = "SELECT count(*) AS high_salary_count FROM kk.salaries WHERE salary > 100000;"
count_high_salary_result_after_index = read_query(connection, count_high_salary_query_after_index)
print("\n启用索引扫描时，薪资高于 100000 的记录数量:", count_high_salary_result_after_index[0][0])

# 分析使用索引的查询计划
explain_index_scan_query = "EXPLAIN ANALYZE SELECT * FROM kk.salaries WHERE salary > 100000;"
explain_index_scan_result = read_query(connection, explain_index_scan_query)
print("\n启用索引扫描时，查询薪资高于 100000 的记录的查询计划:")
for row in explain_index_scan_result:
    print(row[0])

# 删除部分索引
drop_index_query = "DROP INDEX IF EXISTS kk.idx_salaries_high_salary;"
execute_query(connection, drop_index_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
禁用索引扫描时，薪资高于 100000 的记录数量: 94696

禁用索引扫描时，查询薪资高于 100000 的记录的查询计划:
Seq Scan on salaries  (cost=0.00..53782.59 rows=93667 width=24) (actual time=0.109..329.776 rows=94696 loops=1)
  Filter: (salary > 100000)
  Rows Removed by Filter: 2749351
Total runtime: 337.123 ms
Error: relation "idx_salaries_high_salary" already exists

Query executed successfully
Query executed successfully

启用索引扫描时，薪资高于 100000 的记录数量: 94696

启用索引扫描时，查询薪资高于 100000 的记录的查询计划:
Bitmap Heap Scan on salaries  (cost=1796.99..21239.94 rows=96876 width=24) (actual time=16.723..85.225 rows=94696 loops=1)
  Recheck Cond: (salary > 100000)
  Heap Blocks: exact=12351
  ->  Bitmap Index Scan on idx_salaries_high_salary  (cost=0.00..1772.77 rows=96876 width=0) (actual time=13.578..13.578 rows=94696 loops=1)
        Index Cond: (salary > 100000)
Total runtime: 94.553 ms
Query executed successfully
Database connection closed


In [82]:
#(3) 在 dept_emp 表中，查询编号为'd001'的部门中 2001-1-1 之后 的所有员工任职记【复合索引】
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

# 禁用索引扫描
disable_index_scan_query = """
SET enable_indexscan = off;
SET enable_bitmapscan = off;
"""
execute_query(connection, disable_index_scan_query)

# 查询编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量
count_query_without_index = """
SELECT count(*) AS d001_after_2001_count 
FROM kk.dept_emp 
WHERE dept_no = 'd001' AND from_date >= '2001-01-01';
"""
count_result_without_index = read_query(connection, count_query_without_index)
print("禁用索引扫描时，编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量:")
print(count_result_without_index[0][0])

# 分析禁用索引扫描时的查询计划
explain_query_without_index = """
EXPLAIN ANALYZE 
SELECT * FROM kk.dept_emp 
WHERE dept_no = 'd001' AND from_date >= '2001-01-01';
"""
explain_result_without_index = read_query(connection, explain_query_without_index)
print("\n禁用索引扫描时的查询计划:")
for row in explain_result_without_index:
    print(row[0])

# 创建复合索引
create_index_query = "CREATE INDEX idx_dept_emp_dept_no_from_date ON kk.dept_emp(dept_no, from_date);"
execute_query(connection, create_index_query)

# 结束当前事务
connection.commit()

# 开启自动提交模式
connection.autocommit = True
# 分析表
analyze_table_query = "ANALYZE kk.dept_emp;"
execute_query(connection, analyze_table_query)
# 关闭自动提交模式
connection.autocommit = False
# 启用索引扫描
enable_index_scan_query = """
SET enable_indexscan = on;
SET enable_bitmapscan = on;
"""
execute_query(connection, enable_index_scan_query)

# 查询编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量
count_query_with_index = """
SELECT count(*) AS d001_after_2001_count 
FROM kk.dept_emp 
WHERE dept_no = 'd001' AND from_date >= '2001-01-01';
"""
count_result_with_index = read_query(connection, count_query_with_index)
print("\n启用索引扫描时，编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量:")
print(count_result_with_index[0][0])

# 分析启用索引扫描时的查询计划
explain_query_with_index = """
EXPLAIN ANALYZE 
SELECT * FROM kk.dept_emp 
WHERE dept_no = 'd001' AND from_date >= '2001-01-01';
"""
explain_result_with_index = read_query(connection, explain_query_with_index)
print("\n启用索引扫描时的查询计划:")
for row in explain_result_with_index:
    print(row[0])

# 删除复合索引
drop_index_query = "DROP INDEX IF EXISTS kk.idx_dept_emp_dept_no_from_date;"
execute_query(connection, drop_index_query)

if connection:
    connection.close()
    print("Database connection closed")

openGauss Database connection successful
Query executed successfully
禁用索引扫描时，编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量:
630

禁用索引扫描时的查询计划:
Seq Scan on dept_emp  (cost=0.00..7431.05 rows=336 width=25) (actual time=0.099..35.973 rows=630 loops=1)
  Filter: ((from_date >= '2001-01-01 00:00:00'::timestamp without time zone) AND (dept_no = 'd001'::bpchar))
  Rows Removed by Filter: 330973
Total runtime: 36.126 ms
Query executed successfully
Query executed successfully
Query executed successfully

启用索引扫描时，编号为'd001'的部门中 2001-1-1 之后的员工任职记录数量:
630

启用索引扫描时的查询计划:
Bitmap Heap Scan on dept_emp  (cost=11.57..911.63 rows=324 width=25) (actual time=0.301..1.158 rows=630 loops=1)
  Recheck Cond: ((dept_no = 'd001'::bpchar) AND (from_date >= '2001-01-01 00:00:00'::timestamp without time zone))
  Heap Blocks: exact=554
  ->  Bitmap Index Scan on idx_dept_emp_dept_no_from_date  (cost=0.00..11.49 rows=324 width=0) (actual time=0.206..0.206 rows=630 loops=1)
        Index Cond: ((dept_no = 'd001'::bpchar) AND (fr